# Day 34：RAG 评估体系（Ragas 四大核心指标与自动化评测）

> **🎙️ 讲师旁白**：欢迎来到星辰数科《智枢企业级智能体平台》第三阶段 Day 34！今天我们将实战 Ragas 四大核心指标：`Faithfulness`（忠实度）、`Answer Relevancy`（答案相关性）、`Context Precision`（上下文精确率）与 `Context Recall`（上下文召回率），并使用 `RAGEvaluator` 构建自动化评测与诊断流水线（`v0.3.0-day34`）。

## 1. 环境初始化与导入 `project_iterations/day_34` 模块
定位仓库根目录并将 `project_iterations/day_34` 加入 `sys.path`，导入 `EvalSample`、`RAGEvaluator` 及四大指标函数。

In [1]:
import sys
from pathlib import Path

repo_root = Path.cwd()
while not (repo_root / "project_iterations").exists() and repo_root != repo_root.parent:
    repo_root = repo_root.parent
day34_dir = repo_root / "project_iterations" / "day_34"
if str(day34_dir) not in sys.path:
    sys.path.insert(0, str(day34_dir))

from zhishu_agent import (
    EvalSample,
    RAGEvaluator,
    compute_answer_relevancy,
    compute_context_precision,
    compute_context_recall,
    compute_faithfulness,
)

print("已加载模块路径:", day34_dir)

已加载模块路径: /home/antigravity/ab6539763/-AI-Agent/project_iterations/day_34


## 2. 生成侧指标：`Faithfulness`（幻觉检测）与 `Answer Relevancy`（相关性）
对比完全忠实、部分幻觉、数值篡改幻觉以及答非所问场景下的评分变化。

In [2]:
contexts = ["星辰数科年假制度：入职满1年员工每年享有5天带薪年假，满10年享有10天。"]
question = "入职满1年员工每年有多少天带薪年假？"

ans_good = "入职满1年员工每年享有5天带薪年假。"
ans_partial = "入职满1年员工每年享有5天带薪年假。此外公司还提供免费班车。"
ans_numeric_hallucination = "入职满1年员工每年享有15天带薪年假。"
ans_off_topic = "今天北京天气晴朗，适合户外运动。"

print(f"忠实回答 Faithfulness      : {compute_faithfulness(ans_good, contexts):.4f}")
print(f"半幻觉回答 Faithfulness    : {compute_faithfulness(ans_partial, contexts):.4f}")
print(f"数值幻觉 Faithfulness      : {compute_faithfulness(ans_numeric_hallucination, contexts):.4f}")
print(f"切题回答 Answer Relevancy  : {compute_answer_relevancy(question, ans_good):.4f}")
print(f"跑题回答 Answer Relevancy  : {compute_answer_relevancy(question, ans_off_topic):.4f}")

忠实回答 Faithfulness      : 1.0000
半幻觉回答 Faithfulness    : 0.5000
数值幻觉 Faithfulness      : 0.0000
切题回答 Answer Relevancy  : 0.8701
跑题回答 Answer Relevancy  : 0.1320


## 3. 检索侧指标：`Context Precision`（排序敏感）与 `Context Recall`（事实覆盖）
验证相关文档排在首位 vs 垫底时的 `Context Precision` 差异，以及缺失关键事实时的 `Context Recall` 变化。

In [3]:
q_travel = "一线城市差旅酒店住宿报销上限是多少？"
gt_travel = "一线城市酒店住宿报销上限每晚600元。需提供电子发票。"
rel_doc = "差旅报销标准：一线城市酒店住宿报销上限每晚600元，需提供电子发票。"
noise_a = "公司食堂每周三免费提供水果。"
noise_b = "办公用品领用请前往行政前台登记。"

cp_top = compute_context_precision(q_travel, [rel_doc, noise_a, noise_b], gt_travel)
cp_bottom = compute_context_precision(q_travel, [noise_a, noise_b, rel_doc], gt_travel)
print(f"相关文档排第1位 Context Precision: {cp_top:.4f}")
print(f"相关文档排第3位 Context Precision: {cp_bottom:.4f}")

cr_full = compute_context_recall(gt_travel, [rel_doc])
cr_partial = compute_context_recall(gt_travel, ["一线城市酒店住宿报销上限每晚600元。"])
print(f"完整召回 Context Recall        : {cr_full:.4f}")
print(f"部分召回 Context Recall        : {cr_partial:.4f}")

相关文档排第1位 Context Precision: 1.0000
相关文档排第3位 Context Precision: 0.3333
完整召回 Context Recall        : 1.0000
部分召回 Context Recall        : 0.5000


## 4. 使用 `RAGEvaluator` 运行批量数据集评测与逐样本诊断
构造包含健康样本、幻觉风险样本、检索噪音样本的企业测试集，生成 `EvalReport`。

In [4]:
samples = [
    EvalSample(
        sample_id="qa-001",
        question="星辰数科入职满1年员工每年有多少天带薪年假？",
        answer="星辰数科入职满1年员工每年享有5天带薪年假，满10年享有10天。",
        contexts=["星辰数科年假制度：入职满1年员工每年享有5天带薪年假，满10年享有10天。"],
        ground_truth="入职满1年员工每年享有5天带薪年假，满10年享有10天。",
    ),
    EvalSample(
        sample_id="qa-002",
        question="生产环境 Redis 集群禁止执行什么命令？",
        answer="生产环境 Redis 集群禁止执行 KEYS * 命令。此外每天凌晨会自动重启集群。",
        contexts=["生产环境 Redis 集群禁止执行 KEYS * 命令，请使用 SCAN 分批遍历。"],
        ground_truth="生产环境 Redis 集群禁止执行 KEYS * 命令，应使用 SCAN 分批遍历。",
    ),
    EvalSample(
        sample_id="qa-003",
        question="一线城市差旅酒店住宿报销上限是多少？",
        answer="一线城市酒店住宿报销上限为每晚600元，需提供电子发票。",
        contexts=[noise_a, noise_b, rel_doc],
        ground_truth=gt_travel,
    ),
]

evaluator = RAGEvaluator()
report = evaluator.evaluate_dataset(samples)
print(report.format_summary())
print("\n逐样本诊断:")
for item in report.sample_results:
    print(f"  {item.sample_id}: overall={item.overall_score:.4f} | diagnostics={item.diagnostics}")

=== 智枢 RAG 评测报告 (Ragas Metrics) ===
样本总数: 3 | 达标样本: 1 (33.3%)
Faithfulness (忠实度)       : 0.8333
Answer Relevancy (相关性)   : 0.7714
Context Precision (精确率)  : 0.7778
Context Recall (召回率)     : 1.0000
综合均分 (Overall Score)    : 0.8456

逐样本诊断:
  qa-001: overall=0.9603 | diagnostics=['healthy']
  qa-002: overall=0.8273 | diagnostics=['hallucination_risk']
  qa-003: overall=0.7494 | diagnostics=['retrieval_noise']
